In [ ]:
import json
import re
import string
import re


def fix_json_string(json_str):
    try:
        
        if json_str.startswith("Here is the output in JSON format:"):
            json_str = json_str[len("Here is the output in JSON format:"):].strip()
        if json_str.startswith("Here is the output in the required JSON format:"):
            json_str = json_str[len("Here is the output in the required JSON format:"):].strip()

        
        json_str = re.sub(r',\s*}', '}', json_str)
        return json_str
    except Exception as e:
        print(f"Error in fix_json_string: {e}")
        raise


def contains_chinese(text):
    return bool(re.search(r'[\u4e00-\u9fa5]', text))


category_set = set()
style_set = set()
era_set = set()
theme_set = set()
emotion_set = set()


error_indices = []


input_file = "/root/autodl-tmp/crossaug_data/douban_music/douban_music_v11_summary.json"
train_file = "/root/autodl-tmp/crossaug_data/douban_music/crossaug_douban_music_train_.json"
error_output_file = "/root/autodl-tmp/crossaug_data/douban_music/douban_music_error_lines_1.json"
error_index_file = "/root/autodl-tmp/crossaug_data/douban_music/douban_music_error_indices_1.txt"


with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f) 

i = -1
for user in data:
    i += 1
    try:
        response_str = user.get("response", "").replace("\n", "").replace("    ", "")
        fixed_response = fix_json_string(response_str)

   
        if contains_chinese(fixed_response):
            raise ValueError("Response contains Chinese characters")

     
        if '"Category":' in fixed_response:
            category_content = fixed_response.split('"Category":')[1].split('},')[0]
            category_keys = json.loads(category_content + '}').keys()
            category_set.update(category_keys)


        if '"Style":' in fixed_response:
            style_content = fixed_response.split('"Style":')[1].split('},')[0]
            style_keys = json.loads(style_content + '}').keys()
            style_set.update(style_keys)

     
        if '"Era":' in fixed_response:
            era_content = fixed_response.split('"Era":')[1].split('},')[0]
            era_keys = json.loads(era_content + '}').keys()
            era_set.update(era_keys)

        
        if '"Theme":' in fixed_response:
            theme_content = fixed_response.split('"Theme":')[1].split('},')[0]
            theme_keys = json.loads(theme_content + '}').keys()
            theme_set.update(theme_keys)

        
        if '"Emotion":' in fixed_response:
            emotion_content = fixed_response.split('"Emotion":')[1].split('},')[0]
            emotion_keys = json.loads(emotion_content + '}').keys()
            emotion_set.update(emotion_keys)

    except (json.JSONDecodeError, ValueError) as e:
        print(f"Error decoding JSON for user {i}: {e}")
        error_indices.append(i)  

print("Tag files and reasoning file generated successfully.")


with open(train_file, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

error_data = [train_data[idx] for idx in error_indices if idx < len(train_data)]

with open(error_output_file, 'w', encoding='utf-8') as f:
    json.dump(error_data, f, indent=4, ensure_ascii=False)


with open(error_index_file, 'w', encoding='utf-8') as f:
    for idx in error_indices:
        f.write(f"{idx}\n")

print(f"Error data has been saved to {error_output_file}")
print(f"Error indices have been saved to {error_index_file}")


Error decoding JSON for user 105: Response contains Chinese characters
Error decoding JSON for user 119: Response contains Chinese characters
Error decoding JSON for user 352: Response contains Chinese characters
Error decoding JSON for user 382: Response contains Chinese characters
Error decoding JSON for user 418: Response contains Chinese characters
Error decoding JSON for user 621: Response contains Chinese characters
Error decoding JSON for user 663: Expecting ':' delimiter: line 1 column 10 (char 9)
Error decoding JSON for user 719: Response contains Chinese characters
Error decoding JSON for user 752: Response contains Chinese characters
Error decoding JSON for user 795: Response contains Chinese characters
Error decoding JSON for user 920: Response contains Chinese characters
Error decoding JSON for user 961: Response contains Chinese characters
Error decoding JSON for user 983: Response contains Chinese characters
Error decoding JSON for user 1027: Response contains Chinese ch

## 

In [ ]:
import json


summary_file = "/root/autodl-tmp/crossaug_data/douban_music/douban_music_v11_summary.json"

correct_file = "/root/autodl-tmp/crossaug_data/douban_music/douban_music_error_fixed_1.json"

error_index_file = "/root/autodl-tmp/crossaug_data/douban_music/douban_music_error_indices_1.txt"

output_file = "/root/autodl-tmp/crossaug_data/douban_music/douban_music_v11_summary.json"


with open(summary_file, 'r', encoding='utf-8') as f:
    summary_data = json.load(f)


with open(correct_file, 'r', encoding='utf-8') as f:
    correct_data = json.load(f)


with open(error_index_file, 'r', encoding='utf-8') as f:
    error_indices = [int(line.strip()) for line in f.readlines()] 


if len(error_indices) != len(correct_data):
    raise ValueError(f"Mismatch: error lines ({len(error_indices)}) and correct entries ({len(correct_data)})")


for i, line_index in enumerate(error_indices):
    summary_data[line_index] = correct_data[i]  


with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)

print(f"Fixed summary file has been saved to {output_file}")

Fixed summary file has been saved to /root/autodl-tmp/crossaug_data/douban_music/douban_music_v11_summary.json
